imports e variáveis de ambiente

In [1]:
import csv
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()

C:\Users\duda_\AppData\Local\Temp\ipykernel_13184\1166605969.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

carregar a lista de candidatos

In [3]:
with open("../data/candidatos/candidatos.csv", encoding="utf-8-sig") as f:
    candidatos = list(csv.DictReader(f))

print(f"{len(candidatos)} candidatos carregados")   

13 candidatos carregados


carregar os 13 PDFs, já marcando o metadado

In [4]:
documentos = []

for c in candidatos:
    if c["tem_proposta"] != "sim":
        continue

    caminho_pdf = f"../data/candidatos/propostas/{c['id_candidato']}.pdf"
    loader = PyPDFLoader(caminho_pdf)
    paginas = loader.load() # cada página já vem com metadata["page"]

    for pagina in paginas:
        pagina.metadata["candidato"] = c["id_candidato"]
        pagina.metadata["nome_urna"] = c["nome_urna"]
        pagina.metadata["partido"] = c["partido"]

    documentos.extend(paginas)
    print(f"{c['id_candidato']}: {len(paginas)} páginas")

print(f"\nTotal: {len(documentos)} páginas carregadas")

lula: 84 páginas
renan_santos: 51 páginas
hertz_dias: 33 páginas
edmilson_costa: 16 páginas
flavio_bolsonaro: 76 páginas
clariana_barao: 15 páginas
pablo_marcal: 28 páginas
rui_costa_pimenta: 7 páginas


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 29 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 58 0 (offset 0)
Ignoring wrong pointing object 60 0 (offset 0)
Ignoring wrong pointing object 81 0 (offset 0)
Ignoring wrong pointing object 83 0 (offset 0)


zema: 81 páginas
veterinario_wilson_grassi: 58 páginas
ronaldo_caiado: 100 páginas
escritor_augusto_cury: 200 páginas
samara: 67 páginas

Total: 816 páginas carregadas


chunking - o split_documents propaga o metadado automaticamente pra cada fatia

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

fatias = text_splitter.split_documents(documentos)
print(f"{len(fatias)} fatias geradas")
print(fatias[0].metadata) #deve aparecer candidato, nome_urna, partido e page

2454 fatias geradas
{'producer': 'iLovePDF', 'creator': 'Adobe InDesign 21.5 (Windows)', 'creationdate': '2026-08-07T21:58:43-03:00', 'trapped': '/False', 'moddate': '2026-08-08T02:12:53+00:00', 'source': '../data/candidatos/propostas/lula.pdf', 'total_pages': 84, 'page': 0, 'page_label': '1', 'candidato': 'lula', 'nome_urna': 'Lula', 'partido': 'PT'}


PULAR embeddings + Chroma /cota gratuita

In [ ]:
# import time
# import os

# ARQUIVO_PROGRESSO = "candidatos_processados.txt"
# TAMANHO_LOTE = 90

# embeddings = GoogleGenerativeAIEmbeddings(
#     model="models/gemini-embedding-001",
#     task_type="retrieval_document"
# )

# def carregar_progresso():
#     if os.path.exists(ARQUIVO_PROGRESSO):
#         with open(ARQUIVO_PROGRESSO) as f:
#             return set(linha.strip() for linha in f)
#     return set()

# def marcar_concluido(id_candidato):
#     with open(ARQUIVO_PROGRESSO, "a") as f:
#         f.write(id_candidato + "\n")

# processados = carregar_progresso()
# print(f"Já concluídos antes de hoje: {processados or 'nenhum ainda'}")

# vectorstore = Chroma(
#     persist_directory="../data/vectorstores/chroma_candidatos",
#     embedding_function=embeddings,
# )

# for c in candidatos:
#     id_cand = c["id_candidato"]
#     if id_cand in processados:
#         continue

#     fatias_candidato = [f for f in fatias if f.metadata["candidato"] == id_cand]
#     if not fatias_candidato:
#         continue

#     print(f"\nProcessando {id_cand}: {len(fatias_candidato)} fatias")
#     try:
#         for i in range(0, len(fatias_candidato), TAMANHO_LOTE):
#             lote = fatias_candidato[i:i + TAMANHO_LOTE]
#             vectorstore.add_documents(lote)
#             print(f"  {i + len(lote)}/{len(fatias_candidato)}")
#             if i + TAMANHO_LOTE < len(fatias_candidato):
#                 time.sleep(60)
#         marcar_concluido(id_cand)
#         print(f"{id_cand}: concluído")
#     except Exception as e:
#         print(f"\nParou em '{id_cand}' por causa de: {e}")
#         print("Sem problema — rode essa célula de novo amanhã, ela pula quem já terminou.")
#         break

# print("\nFim da execução de hoje.")

5. embeddings local + Chroma /trade off

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings #perguntar
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2" #modelo de embeddings
)

vectorstore = Chroma.from_documents(
    documents=fatias,
    embedding=embeddings,
    persist_directory="../data/vectorstores/chroma_candidatos",
)

print("Banco vetorial criado com embeddings locais!")

Banco vetorial criado com embeddings locais!


6. teste com filtro por candidato

In [11]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4, "filter": {"candidato": "renan_santos"}}
)

resultados = retriever.invoke("Quais são as propostas para educação?")
for r in resultados:
    print(f"[{r.metadata['candidato']} - página {r.metadata['page'] +1}]")
    print(r.page_content[:200])
    print("---")

[renan_santos - página 31]
sociedade. Em um país de tradição universitária incipiente como o Brasil, que ainda possui um perfil marca-
damente elitista, a maior parte da sociedade possui somente esta escolaridade em seu limite.
---
[renan_santos - página 31]
portamental do estudante, com implicações punitivas, de modo a restaurar o sentido disciplinar dentro das 
escolas. Precisaremos abrir diálogo amplo com a comunidade pedagógica para que esta admita o 
---
[renan_santos - página 32]
LIVRO AMARELO - MISSÃO 2026
32
1   Elevar o nível de qualidade de nossas escolas, com foco especial nas disciplinas básicas do currículo 
(língua portuguesa e matemática), assegurando formação sólida 
---
[renan_santos - página 30]
LIVRO AMARELO - MISSÃO 2026
30
 
FORMAR ELITES, CULTIVAR O POVO
Educação
O PROBLEMA EM NÚMEROS
A RESPOSTA DA MISSÃO
O PROBLEMA
A
quele velho truísmo que diz que a educação forma nações é uma verdade a
---
